**UTS Data Mining II · Audio / Signal**

Pilih satu input dan hapus sisanya sebelum Run All. Ganti path sesuai soal.

- Folder/ZIP: jalankan bagian audio sampai ekstraksi fitur; hapus kedua bagian NPZ.
- NPZ: hapus **Baca folder audio** sampai sebelum **Baca NPZ Signal**.

Pertahankan **Import**, lalu lanjut ke **Standardisasi**.

In [ ]:
# Jalankan sekali jika diperlukan:
# %pip install numpy pandas matplotlib scipy scikit-learn librosa soundfile openpyxl

**Import**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import spectrogram
from IPython.display import Audio, display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

**Baca folder audio**

In [ ]:
data_dir = Path("/content/data/audio")
extensions = (".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aiff", ".aif")
files = sorted([p for p in data_dir.rglob("*") if p.suffix.lower() in extensions])

df = pd.DataFrame({"path": [str(p) for p in files]})
print("Jumlah audio:", len(df))
display(df.head())

**Baca ZIP audio**

Ekstrak ke folder baru; subfolder ikut dibaca. Lanjut ke **Baca satu audio**.

In [ ]:
import zipfile

zip_path = "/content/audio.zip"
data_dir = Path("/content/audio_extracted")
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(data_dir)

extensions = (".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aiff", ".aif")
files = sorted([p for p in data_dir.rglob("*") if p.suffix.lower() in extensions])

df = pd.DataFrame({"path": [str(p) for p in files]})
print("Jumlah audio:", len(df))
display(df.head())

**Baca satu audio**

Preview mono dengan sample rate asli (`sr=None`).

In [ ]:
import librosa

signal, sr = librosa.load(files[0], sr=None, mono=True)

print("File:", files[0])
print("Sample rate:", sr)
print("Shape array:", signal.shape)
print("Durasi:", len(signal) / sr, "detik")
print("10 sampel pertama:", signal[:10])

**Waveform dan putar audio**

In [ ]:
time = np.arange(len(signal)) / sr

plt.figure(figsize=(10, 4))
plt.plot(time, signal)
plt.xlabel("Waktu (detik)")
plt.ylabel("Amplitudo")
plt.title("Waveform audio")
plt.show()

display(Audio(signal, rate=sr))

**Spectrogram**

In [ ]:
frequencies, times, power = spectrogram(signal, fs=sr)

plt.figure(figsize=(10, 4))
plt.pcolormesh(times, frequencies, 10 * np.log10(power + 1e-12), shading="auto")
plt.xlabel("Waktu (detik)")
plt.ylabel("Frekuensi (Hz)")
plt.title("Spectrogram audio")
plt.colorbar(label="PSD (dB)")
plt.tight_layout()
plt.show()

**Preprocessing dan ekstraksi fitur**

Mono, 22.050 Hz, trim senyap; ambil rata-rata 13 MFCC dan 5 fitur audio lainnya.

In [ ]:
features = []

for file in files:
    y, sr = librosa.load(file, sr=22050, mono=True)
    y, _ = librosa.effects.trim(y)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    zcr = librosa.feature.zero_crossing_rate(y)
    rms = librosa.feature.rms(y=y)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)

    feature = np.hstack([mfcc.mean(axis=1), zcr.mean(), rms.mean(), centroid.mean(), bandwidth.mean(), rolloff.mean()])
    features.append(feature)

In [ ]:
feature_columns = [f"mfcc_{i + 1}" for i in range(13)]
feature_columns += ["zcr", "rms", "centroid", "bandwidth", "rolloff"]

df_features = pd.DataFrame(features, columns=feature_columns)
display(df_features.head())
display(df_features.describe())
print("Shape fitur:", df_features.shape)

**Baca NPZ Signal**

`x_train`: sinyal 128 titik; `y_train`: label. Contoh memakai 2.000 train (ubah `n`); test tetap terpisah. Fitur memakai 128 amplitudo dan sampling tetap 128 Hz. Label hanya untuk evaluasi.

In [ ]:
data = np.load("/content/Data Latihan Datmin II (Signal).npz", allow_pickle=False)
print("Nama array:", data.files)

n = 2000
signals = data["x_train"][:n].astype(np.float32)
df = pd.DataFrame({"label": data["y_train"][:n]})
sr = int(data["sample_rate_hz"].item())
time = data["time_seconds"]
class_names = data["class_names"]

feature_columns = [f"point_{i}" for i in range(signals.shape[1])]
df_features = pd.DataFrame(signals, columns=feature_columns)
print("Shape sinyal:", signals.shape, "| Sample rate:", sr, "Hz")
display(df_features.head())

**Contoh sinyal NPZ**

Plot sinyal, lalu lanjut ke **Standardisasi**. Fitur amplitudo sensitif terhadap fase/pergeseran sinyal.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(time, signals[0])
plt.title(class_names[df.loc[0, "label"]])
plt.xlabel("Waktu (detik)")
plt.ylabel("Amplitudo relatif")
plt.show()

**Standardisasi**

Samakan skala fitur dengan StandardScaler.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features)
print("Shape fitur standar:", X_scaled.shape)

**Modeling dan evaluasi K-Means**

Coba beberapa `k`, masing-masing minimal 2 dan lebih kecil dari jumlah data. Silhouette memakai maksimal 3.000 data sebagai perkiraan skor.

In [ ]:
k_list = [3, 5, 7]
scores = []
inertia = []

for k in k_list:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels, sample_size=min(3000, len(labels)), random_state=42)
    scores.append(score)
    inertia.append(model.inertia_)
    print(f"K={k}, silhouette={score:.4f}")

df_scores = pd.DataFrame({"k": k_list, "silhouette": scores, "inertia": inertia})
display(df_scores)

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(k_list, scores, marker="o")
plt.xlabel("K")
plt.ylabel("Silhouette score")
plt.title("Silhouette")

plt.subplot(1, 2, 2)
plt.plot(k_list, inertia, marker="o")
plt.xlabel("K")
plt.ylabel("Inertia")
plt.title("Elbow")
plt.tight_layout()
plt.show()

**Model akhir**

Ganti `k = 7` sesuai hasil evaluasi atau soal.

In [ ]:
k = 7
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)
display(df.head())
display(df["cluster"].value_counts().sort_index())

**Visualisasi cluster dengan PCA**

PCA hanya untuk grafik; clustering tetap memakai fitur sebelumnya.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
for cluster in range(k):
    mask = df["cluster"] == cluster
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f"Cluster {cluster}", alpha=0.6)
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("Cluster audio/sinyal")
plt.legend()
plt.show()

**Profil fitur tiap cluster**

In [ ]:
df_profile = df_features.copy()
df_profile["cluster"] = df["cluster"].to_numpy()
display(df_profile.groupby("cluster").mean().round(3))

**Contoh audio tiap cluster**

In [ ]:
for cluster in range(k):
    print("Cluster", cluster)
    display(df.loc[df["cluster"] == cluster].head(3))

**Simpan hasil (opsional)**

Hapus `#` pada kode untuk menyimpan CSV.

In [ ]:
# df.to_csv("audio_hasil.csv", index=False)
# df_features.to_csv("audio_fitur.csv", index=False)
# df_scores.to_csv("audio_skor_k.csv", index=False)

**Alternatif input**
Pakai satu loader saja; hapus input lainnya sebelum Run All.

**Kaggle** — install `kagglehub`; ganti `data_dir` pada loader folder.
```python
import kagglehub
folder = Path(kagglehub.dataset_download("kinguistics/heartbeat-sounds"))
print(list(folder.iterdir()))
data_dir = folder / "set_b"
```

**CSV lokasi audio** — ganti loader folder. `filename` berisi path absolut atau relatif terhadap `data_dir`.
```python
data_dir = Path("/content/data/audio")
df = pd.read_csv("/content/audio_paths.csv")
files = [data_dir / str(name) for name in df["filename"]]
df["path"] = [str(p) for p in files]
df = df.reset_index(drop=True)
```
Excel: gunakan `pd.read_excel(...)`.

**CSV fitur** — ganti bagian loader sampai sebelum **Standardisasi**. Pilih kolom fitur; keluarkan label/ID, lalu lanjut ke **Standardisasi**.
```python
df = pd.read_csv("/content/audio_features.csv")
feature_columns = ["mfcc_1", "mfcc_2", "rms"]
df_features = df[feature_columns].copy()
display(df_features.head())
```

**Evaluasi label (opsional)**
Jalankan setelah **Model akhir**; sesuaikan kolom `label`.
```python
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
print("ARI:", adjusted_rand_score(df["label"], df["cluster"]))
print("NMI:", normalized_mutual_info_score(df["label"], df["cluster"]))
```
Periksa contoh anggota sebelum memberi arti pada cluster.
Untuk soal waveform/spectrogram saja, berhenti setelah spectrogram.